In [ ]:
import climate_indices
import climate_library
from climate_library.climate_index import ClimateIndex

In [ ]:
import xclim
import xclim.indices
import xclim.core.units as xu
from xclim.testing import open_dataset
from xclim.indices import standardized_precipitation_index
from xclim.indices.stats import standardized_index_fit_params
from xclim.core.calendar import percentile_doy

In [ ]:
import pandas as pd
import geopandas as gpd

In [ ]:
from netCDF4 import Dataset

In [ ]:
import xarray as xr
import numpy as np
from datetime import datetime
from scipy import stats as st
from tqdm import tqdm
import json

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [ ]:
import climate_V2

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
ds = xr.open_dataset("netcdf.nc")
ds = ds.rename({'oldname', 'newname'})
ds.to_netcdf('newnewcdf.nc')

In [ ]:
ds = xr.open_dataset("netcdf.nc")
ds = ds.rename(oldname='newname')
ds.to_netcdf('newnewcdf.nc')

In [ ]:
ds = open_dataset(precipitation)
ds['tp'].attrs['units'] = 'mm/day'
ds['tp'].values*1000
ds.to_netcdf('convert_precipitation.nc')

In [ ]:
import netCDF4 as nc

# Open the NetCDF file in read/write mode
dataset = nc.Dataset('your_file.nc', 'r+')

# Rename the coordinate
dataset.renameDimension('valid_time', 'time')

# Save the changes
dataset.close()

In [ ]:
import netCDF4 as nc

# Open the NetCDF file in read/write mode
dataset = nc.Dataset('your_file.nc', 'r+')

# Rename the coordinate
dataset.renameVariable('valid_time', 'time')

# Save the changes
dataset.close()

In [ ]:
import netCDF4 as nc

# Open the NetCDF file in read/write mode
dataset = nc.Dataset('your_file.nc', 'r+')

# Transpose the data variable to a coordinate
dataset.transpose('valid_time', 'time')

# Save the changes
dataset.close()

In [ ]:
# Standardized Precipitation Index Function
def spi(ds, thresh, dimension):
    #ds - data ; thresh - time interval / scale; dimension - dimension as a string

    #Rolling Mean / Moving Averages
    ds_ma = ds.rolling(time = thresh, center=False).mean(dim=dimension)

    #Natural log of moving averages
    ds_In = np.log(ds_ma)
    ds_In = ds_In.where(np.isinf(ds_In) == False) #= np.nan  #Change infinity to NaN

    #Overall Mean of Moving Averages
    ds_mu = ds_ma.mean(dimension)

    #Summation of Natural log of moving averages
    ds_sum = ds_In.sum(dimension)

    #Computing essentials for gamma distribution
    n = ds_In[thresh-1:, :, :].count(dimension)                  #size of data

    A = np.log(ds_mu) - (ds_sum/n)             #Computing A
    alpha = (1/(4*A))*(1+(1+((4*A)/3))**0.5)   #Computing alpha  (a)
    beta = ds_mu/alpha                         #Computing beta (scale)
    
    #Gamma Distribution (CDF) 
    gamma_func = lambda data, a, scale: st.gamma.cdf(data, a=a, scale=scale)
    gamma = xr.apply_ufunc(gamma_func, ds_ma, alpha, beta)
    
    #Standardized Precipitation Index   (Inverse of CDF)
    norminv = lambda data: st.norm.ppf(data, loc=0, scale=1)
    norm_spi = xr.apply_ufunc(norminv, gamma)  #loc is mean and scale is standard dev.
    
    return ds_ma, ds_In , ds_mu, ds_sum,n, A, alpha, beta, gamma, norm_spi

da_data = xr.open_dataset('C:/Netcdf/cru_ts4.08.1901.2023.pre.dat.nc')
ds_RR = da_data['pre']
# ds_RR_Thailand= ds_RR.sel(lon=slice(96, 106), lat=slice(4, 21),time=slice('2015','2018'))
# ds_RR_Thailand= ds_RR.sel(lon=slice(96, 106), lat=slice(4, 21),time=slice('1901', '1902'))
ds_RR_Thailand= ds_RR.sel(lon=slice(96, 106), lat=slice(4, 21),time='1901')
i=3
ddata = spi(ds_RR_Thailand,i,'time')[9]
ddata.plot(cmap='RdBu', col='time', col_wrap=4, vmin=-2.5, vmax=2.5)

# print(ddata)
plt.show()

In [ ]:
#Standardized Precipitation Index Function
def spi(ds, thresh):
    #ds - data ; thresh - time interval / scale
    
    #Rolling Mean / Moving Averages
    ds_ma = ds.rolling(thresh, center=False).mean()
    
    #Natural log of moving averages
    ds_In = np.log(ds_ma)
    ds_In[ np.isinf(ds_In) == True] = np.nan  #Change infinity to NaN
    
    #Overall Mean of Moving Averages
    ds_mu = np.nanmean(ds_ma)
    
    #Summation of Natural log of moving averages
    ds_sum = np.nansum(ds_In)
        
    #Computing essentials for gamma distribution
    n = len(ds_In[thresh-1:])                  #size of data
    A = np.log(ds_mu) - (ds_sum/n)             #Computing A
    alpha = (1/(4*A))*(1+(1+((4*A)/3))**0.5)   #Computing alpha  (a)
    beta = ds_mu/alpha                         #Computing beta (scale)
    
    #Gamma Distribution (CDF)
    gamma = st.gamma.cdf(ds_ma, a=alpha, scale=beta)  
    
    #Standardized Precipitation Index   (Inverse of CDF)
    norm_spi = st.norm.ppf(gamma, loc=0, scale=1)  #loc is mean and scale is standard dev.
    
    return ds_ma, ds_In, ds_mu, ds_sum, n, A, alpha, beta, gamma, norm_spi

data = pd.read_csv('precipitation.csv', usecols=[1])

data = data.set_index(pd.date_range('1901', '2024', freq='M'))
times = [3, 6, 9, 12, 24]
for i in times:
    x = spi(data['station1-97.46153389044547-18.4142241806728'], i)
    data['spi_'+str(i)] = x[9]

fig, axes = plt.subplots(nrows=5, figsize=(15, 10))
plt.subplots_adjust(hspace=0.15)
for i, ax in enumerate(axes):
    col_scheme=np.where(data['spi_'+str(times[i])]>0, 'b','r')

    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.bar(data.index, data['spi_'+str(times[i])], width=25, align='center', color=col_scheme, label='SPI '+str(times[i]))
    ax.axhline(y=0, color='k')
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.legend(loc='upper right')
    ax.set_yticks(range(-3,4), range(-3,4))
    ax.set_ylabel('SPI', fontsize=12)
    
    if i<len(times)-1:
        ax.set_xticks([],[])

plt.show()

In [ ]:
da_data = xr.open_dataset('C:/Netcdf/cru_ts4.08.1901.2023.pre.dat.nc')
ds_RR = da_data['pre']

ds_RR_Thailand= ds_RR.sel(lon=slice(96, 106), lat=slice(4, 21),time='1901')
i=3

test = climate_V2.Climate(ds_RR_Thailand)
ddata = test.calculate_spi(thresh=i,dimension='time',precip_var='pre')

ddata[9].plot(cmap='RdBu', col='time', col_wrap=4, vmin=-2.5, vmax=2.5)
plt.show()

RX1day version 1

In [ ]:
import json
import xclim
import pandas as pd
import geopandas as gpd
from tqdm import tqdm
import xarray as xr
import numpy as np
from shapely.geometry import mapping
from province import province_coord

precipitation = 'C:/Netcdf/TH_precipitation_day_1960-2022.nc'
shapefile = gpd.read_file('./src/Geo-data/thailand-Geo.json')

def create_grid_polygon(lon_center, lat_center, lon_step, lat_step):
    return [
        [float(lon_center - lon_step / 2), float(lat_center - lat_step / 2)],  # down left corner 
        [float(lon_center + lon_step / 2), float(lat_center - lat_step / 2)],  # down right corner
        [float(lon_center + lon_step / 2), float(lat_center + lat_step / 2)],  # up right corner
        [float(lon_center - lon_step / 2), float(lat_center + lat_step / 2)],  # up left corner 
        [float(lon_center - lon_step / 2), float(lat_center - lat_step / 2)]   # corner 1
    ]


def calculate_weighted_polygon(province_name, shapefile, data, cru):
    province_coord = shapefile[shapefile['NAME_1'] == province_name]
    
    if province_coord.empty:
        print(f"No data in province: {province_name}")
        return None, None  # Return None if no data
    
    grid_in_province = data[data.geometry.intersects(province_coord.geometry.union_all())]
    
    province_area = province_coord.geometry.union_all().area
    
    total_weighted = 0
    total_percentage = 0
    
    for idx, grid in grid_in_province.iterrows():

        intersection_area = grid.geometry.intersection(province_coord.geometry.union_all()).area
        
        intersection_percentage_of_province = (intersection_area / province_area) * 100
        if(len(cru) != ''):
            grid_value = grid[cru]
        else:
            print("You not select something you need to try again")
            break

        grid_value = np.nan_to_num(grid_value, nan=0.0)
        
        weighted_temp = grid_value * intersection_percentage_of_province
        total_weighted += weighted_temp
        total_percentage += intersection_percentage_of_province
    
    average_value = total_weighted / total_percentage #if total_percentage != 0 else None
    return average_value, province_coord.geometry

pr = xr.open_dataset(precipitation)
pre = pr.tp

rx1day = xclim.indices.max_1day_precipitation_amount(pre, freq="YS")

lon = rx1day['longitude']
# print(lon)
lat =  rx1day['latitude']
# print(lat)
lon_step = float(lon[1]-lon[0])

lat_step = float(lat[1]-lat[0])

# print(rx1day.sel(time='1960-01-01'))
for date in tqdm(rx1day['time'].values, leave=False, ascii=False, ncols=100):
    
    year = str(date)[0:4]
    
    _rx1day = rx1day.sel(time=year)

    # thai_grid = _rx1day.clip(shapefile)
    # thai_grid.plot(cmap='jet')
    # _rx1day.plot(cmap='jet')

    # lon = _rx1day['longitude']
    # lat =  _rx1day['latitude']

    # lon_step = float(lon[1]-lon[0])
    # lat_step = float(lat[1]-lat[0])

    features = []
    rx1day_value = _rx1day.values[0]

    for i, lon_value in tqdm(enumerate(lon), desc="Create longitude grid...",unit=' grid', leave=False, ncols=75):#i = 48
        for j, lat_value in tqdm(enumerate(lat), desc="Create latitude  grid...",unit=' grid', leave=False, ncols=75):# j = 72
            rx = rx1day_value[j, i]
            if not pd.isnull(rx):
                grid_polygon = create_grid_polygon(lon_value, lat_value, lon_step, lat_step)
                features.append({
                        "type": "Feature",
                        "geometry": {
                            "type": "Polygon",
                            "coordinates": [grid_polygon]
                        },
                        "properties": {
                            "year": str(year),
                            "rx1day": float(rx) 
                        }
                    })
                
    geojson_data = {
        "type": "FeatureCollection",
        "features": features
    }

    data = gpd.GeoDataFrame.from_features(geojson_data['features'])

    # print(data)

    geojson_data = {
        "type": "FeatureCollection",
        "features": []
    }
    
    for region in tqdm(province_coord(), desc=f"Loading region & province {date}...", ascii=False, ncols=150, colour='yellow'): 
        for province in region:
            name, geometry, region_name = province
            avg_rx, province_shape = calculate_weighted_polygon(name, shapefile, data, cru='rx1day')
            if avg_rx is not None:
                feature = {
                    "type": "Feature",
                    "geometry": mapping(geometry),  
                    "properties": {
                        "name": name,
                        "region": region_name,
                        "year": year,
                        "rx1day":float(avg_rx)
                    }
                }
                geojson_data["features"].append(feature)

    output_geojson_path = f'./src/Geo-data/Year-Dataset/rx1day_{year}.json'
    with open(output_geojson_path, 'w', encoding='utf-8') as geojson_file:
        json.dump(geojson_data, geojson_file, indent=2, ensure_ascii=False)
    print(output_geojson_path + " is saved.")

print("\nGeoJSON file polygon saved complete.")

RX1day version 2

In [ ]:
import json
import xclim
import pandas as pd
import geopandas as gpd
from tqdm import tqdm
import xarray as xr
import numpy as np
from shapely.geometry import mapping
from province import province_coord

precipitation = 'C:/Netcdf/TH_precipitation_day_1960-2022.nc'
shapefile = gpd.read_file('./src/Geo-data/thailand-Geo.json')

def create_grid_polygon(lon_center, lat_center, lon_step, lat_step):
    return [
        [float(lon_center - lon_step / 2), float(lat_center - lat_step / 2)],  # down left corner 
        [float(lon_center + lon_step / 2), float(lat_center - lat_step / 2)],  # down right corner
        [float(lon_center + lon_step / 2), float(lat_center + lat_step / 2)],  # up right corner
        [float(lon_center - lon_step / 2), float(lat_center + lat_step / 2)],  # up left corner 
        [float(lon_center - lon_step / 2), float(lat_center - lat_step / 2)]   # corner 1
    ]


def calculate_weighted_polygon(province_name, shapefile, data, cru):
    province_coord = shapefile[shapefile['NAME_1'] == province_name]
    
    if province_coord.empty:
        print(f"No data in province: {province_name}")
        return None, None  # Return None if no data
    
    grid_in_province = data[data.geometry.intersects(province_coord.geometry.union_all())]
    
    province_area = province_coord.geometry.union_all().area
    
    total_weighted = 0
    total_percentage = 0
    
    for idx, grid in grid_in_province.iterrows():

        intersection_area = grid.geometry.intersection(province_coord.geometry.union_all()).area
        
        intersection_percentage_of_province = (intersection_area / province_area) * 100
        if(len(cru) != ''):
            grid_value = grid[cru]
        else:
            print("You not select something you need to try again")
            break

        grid_value = np.nan_to_num(grid_value, nan=0.0)
        
        weighted_temp = grid_value * intersection_percentage_of_province
        total_weighted += weighted_temp
        total_percentage += intersection_percentage_of_province
    
    average_value = total_weighted / total_percentage #if total_percentage != 0 else None
    return average_value, province_coord.geometry

pr = xr.open_dataset(precipitation)
pre = pr.tp

rx1day = xclim.indices.max_1day_precipitation_amount(pre, freq="YS")

lon = rx1day['longitude']

lat =  rx1day['latitude']

lon_step = float(lon[1]-lon[0])

lat_step = float(lat[1]-lat[0])

features = []
# print(rx1day.sel(time='1960-01-01'))
for date in tqdm(rx1day['time'].values, leave=False, ascii=False, ncols=100):
    
    year = str(date)[0:4]
    
    _rx1day = rx1day.sel(time=year)

    # thai_grid = _rx1day.clip(shapefile)
    # thai_grid.plot(cmap='jet')
    # _rx1day.plot(cmap='jet')

    # lon = _rx1day['longitude']
    # lat =  _rx1day['latitude']

    # lon_step = float(lon[1]-lon[0])
    # lat_step = float(lat[1]-lat[0])

    
    rx1day_value = _rx1day.values[0]

    for i, lon_value in tqdm(enumerate(lon), desc="Create longitude grid...",unit=' grid', leave=False, ncols=75):#i = 48
        for j, lat_value in tqdm(enumerate(lat), desc="Create latitude  grid...",unit=' grid', leave=False, ncols=75):# j = 72
            rx = rx1day_value[j, i]
            if not pd.isnull(rx):
                grid_polygon = create_grid_polygon(lon_value, lat_value, lon_step, lat_step)
                features.append({
                        "type": "Feature",
                        "geometry": {
                            "type": "Polygon",
                            "coordinates": [grid_polygon]
                        },
                        "properties": {
                            "year": str(year),
                            "rx1day": float(rx) 
                        }
                    })
            
geojson_data = {
    "type": "FeatureCollection",
    "features": features
}

data = gpd.GeoDataFrame.from_features(geojson_data['features'])

geojson_data = {
    "type": "FeatureCollection",
    "features": []
}

for date in tqdm(rx1day['time'].values, leave=False, ascii=False, ncols=100):
    
    year = str(date)[0:4]

    data_year = data[data['year'] == year]
    

    for region in tqdm(province_coord(), desc=f"Loading region & province {date}...", ascii=False, ncols=150, colour='yellow'): 
        for province in region:
            name, geometry, region_name = province
            avg_rx, province_shape = calculate_weighted_polygon(name, shapefile, data_year, cru='rx1day')
            if avg_rx is not None:
                feature = {
                    "type": "Feature",
                    "geometry": mapping(geometry),  
                    "properties": {
                        "name": name,
                        "region": region_name,
                        "year": year,
                        "rx1day":float(avg_rx)
                    }
                }
                geojson_data["features"].append(feature)

output_geojson_path = f'./src/Geo-data/Year-Dataset/rx1day.json'
with open(output_geojson_path, 'w', encoding='utf-8') as geojson_file:
    json.dump(geojson_data, geojson_file, indent=2, ensure_ascii=False)
print(output_geojson_path + " is saved.")

print("\nGeoJSON file polygon saved complete.")

RX1day version 3

In [ ]:
import xarray as xr
import pandas as pd
import json
import xclim
import warnings
warnings.filterwarnings('ignore')


ds_tmax = xr.open_dataset("C:/Netcdf/TH_tmax_ERA5_day.1960-2022.nc")
ds_tmin = xr.open_dataset("C:/Netcdf/TH_tmin_ERA5_day.1960-2022.nc")
ds_pr = xr.open_dataset("C:/Netcdf/TH_precipitation_day_1960-2022.nc")
# ds_pr = xr.open_dataset("C:/Netcdf/convert_precipitation.nc")


def create_grid_polygon(lon_center, lat_center, lon_step, lat_step):
    return [
        [float(lon_center - lon_step / 2), float(lat_center - lat_step / 2)],  # มุมล่างซ้าย
        [float(lon_center + lon_step / 2), float(lat_center - lat_step / 2)],  # มุมล่างขวา
        [float(lon_center + lon_step / 2), float(lat_center + lat_step / 2)],  # มุมบนขวา
        [float(lon_center - lon_step / 2), float(lat_center + lat_step / 2)],  # มุมบนซ้าย
        [float(lon_center - lon_step / 2), float(lat_center - lat_step / 2)]   # ปิดกรอบ
    ]

# คำนวณระยะห่างระหว่างพิกัด (step)
lon_step = float(ds_tmax['longitude'][1] - ds_tmax['longitude'][0])
lat_step = float(ds_tmax['latitude'][1] - ds_tmax['latitude'][0])

for year in range(1960, 1961):

    data_tmax_year = ds_tmax.sel(time=str(year))
    data_tmin_year = ds_tmin.sel(time=str(year))
    data_pr_year = ds_pr.sel(time=str(year))
    tmax_monthly_mean = (data_tmax_year['mx2t'].resample(time='M').mean() - 273.15)

    tmin_monthly_mean = (data_tmin_year['mn2t'].resample(time='M').mean() - 273.15)
    pr_monthly_sum = data_pr_year['tp'].resample(time='M').sum() * 1000
    # print(data_pr_year['tp'])
    # data_pr_year['tp']*1000
    pre = data_pr_year.tp*1000
    # print(pre)
    # print(pr_monthly_sum)
    rx1day = xclim.indices.max_1day_precipitation_amount(pre, freq='ME')

    features = []
    lon, lat = tmax_monthly_mean['longitude'].values, tmax_monthly_mean['latitude'].values

    for month_idx, month in enumerate(tmax_monthly_mean['time'].values):
        tmax_values = tmax_monthly_mean.isel(time=month_idx).values
        tmin_values = tmin_monthly_mean.isel(time=month_idx).values
        pr_values = pr_monthly_sum.isel(time=month_idx).values
        rx1day_values = rx1day.isel(time=month_idx).values

        txx = tmax_values.max()
        tnn = tmin_values.min()

        for i, lon_value in enumerate(lon):
            for j, lat_value in enumerate(lat):
                tmax = tmax_values[j, i]
                tmin = tmin_values[j, i]
                pr = pr_values[j, i]
                rx = rx1day_values[j, i]
                if not pd.isnull(tmax) and not pd.isnull(tmin) and not pd.isnull(pr):
                    grid_polygon = create_grid_polygon(lon_value, lat_value, lon_step, lat_step)
                    features.append({
                        "type": "Feature",
                        "geometry": {
                            "type": "Polygon",
                            "coordinates": [grid_polygon]
                        },
                        "properties": {
                            "tmax": float(tmax),
                            "tmin": float(tmin),
                            "pre": float(pr),
                            "txx": float(txx),
                            "tnn": float(tnn),
                            "rx1day" :float(rx),
                            "month": pd.Timestamp(month).month
                        }
                    })

    geojson_data = {
        "type": "FeatureCollection",
        "features": features
    }

    output_file = f"./src/Geo-data/Era-Dataset/era_data_grid_{year}.json"
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(geojson_data, f, ensure_ascii=False, indent=4)

    print(f"Data year {year} has been saved file in folder {output_file}")